In [0]:
path = "/Volumes/workspace/default/credit/UCI_Credit_Card.csv"  # volume path 
df = spark.read.csv(path, header=True, inferSchema=True)

df = (df.withColumnRenamed("default.payment.next.month", "label")
        .withColumnRenamed("PAY_0", "PAY_1"))

print(df.count(), "rows")
df.printSchema()
display(df.limit(5))

30000 rows
root
 |-- ID: integer (nullable = true)
 |-- LIMIT_BAL: double (nullable = true)
 |-- SEX: integer (nullable = true)
 |-- EDUCATION: integer (nullable = true)
 |-- MARRIAGE: integer (nullable = true)
 |-- AGE: integer (nullable = true)
 |-- PAY_1: integer (nullable = true)
 |-- PAY_2: integer (nullable = true)
 |-- PAY_3: integer (nullable = true)
 |-- PAY_4: integer (nullable = true)
 |-- PAY_5: integer (nullable = true)
 |-- PAY_6: integer (nullable = true)
 |-- BILL_AMT1: double (nullable = true)
 |-- BILL_AMT2: double (nullable = true)
 |-- BILL_AMT3: double (nullable = true)
 |-- BILL_AMT4: double (nullable = true)
 |-- BILL_AMT5: double (nullable = true)
 |-- BILL_AMT6: double (nullable = true)
 |-- PAY_AMT1: double (nullable = true)
 |-- PAY_AMT2: double (nullable = true)
 |-- PAY_AMT3: double (nullable = true)
 |-- PAY_AMT4: double (nullable = true)
 |-- PAY_AMT5: double (nullable = true)
 |-- PAY_AMT6: double (nullable = true)
 |-- label: integer (nullable = true)



ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,label
1,20000.0,2,2,1,24,2,2,-1,-1,-2,-2,3913.0,3102.0,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
2,120000.0,2,2,2,26,-1,2,0,0,0,2,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
3,90000.0,2,2,2,34,0,0,0,0,0,0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
4,50000.0,2,2,1,37,0,0,0,0,0,0,46990.0,48233.0,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
5,50000.0,1,2,1,57,-1,0,-1,0,0,0,8617.0,5670.0,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [0]:
from pyspark.sql import functions as F

# Class balance
display(df.groupBy("label").count())

# Fix undocumented category codes
df = (df
  .withColumn("EDUCATION", F.when(F.col("EDUCATION").isin(0,5,6), 4).otherwise(F.col("EDUCATION")))
  .withColumn("MARRIAGE", F.when(F.col("MARRIAGE")==0, 3).otherwise(F.col("MARRIAGE"))))

# Missing values check
display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]))

df.createOrReplaceTempView("credit")

label,count
1,6636
0,23364


ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,label
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
%sql
SELECT PAY_1, COUNT(*) AS customers, ROUND(AVG(label)*100,1) AS default_rate_pct
FROM credit GROUP BY PAY_1 ORDER BY PAY_1

PAY_1,customers,default_rate_pct
-2,2759,13.2
-1,5686,16.8
0,14737,12.8
1,3688,33.9
2,2667,69.1
3,322,75.8
4,76,68.4
5,26,50.0
6,11,54.5
7,9,77.8


In [0]:
%sql
SELECT CASE WHEN AGE<30 THEN '<30' WHEN AGE<40 THEN '30-39' WHEN AGE<50 THEN '40-49' ELSE '50+' END AS age_band,
       ROUND(AVG(label)*100,1) AS default_rate_pct
FROM credit GROUP BY 1 ORDER BY 1

age_band,default_rate_pct
30-39,20.3
40-49,23.0
50+,25.3
<30,22.8


In [0]:
bill   = [f"BILL_AMT{i}" for i in range(1,7)]
paid   = [f"PAY_AMT{i}" for i in range(1,7)]
status = [f"PAY_{i}" for i in range(1,7)]

df = (df
  .withColumn("avg_bill", sum(F.col(c) for c in bill)/6)
  .withColumn("avg_payment", sum(F.col(c) for c in paid)/6)
  .withColumn("pay_to_bill_ratio", F.col("avg_payment")/(F.abs(F.col("avg_bill"))+1))
  .withColumn("utilisation", F.col("BILL_AMT1")/F.col("LIMIT_BAL"))
  .withColumn("late_count", sum(F.when(F.col(c) > 0, 1).otherwise(0) for c in status))
  .withColumn("label", F.col("label").cast("double")))

display(df.select("avg_bill","pay_to_bill_ratio","utilisation","late_count","label").limit(5))

avg_bill,pay_to_bill_ratio,utilisation,late_count,label
1284.0,0.0893644617380026,0.19565,2,1.0
2846.1666666666665,0.2926886378270796,0.02235,2,1.0
16942.166666666668,0.10838194355639932,0.32487777777777777,0,0.0
38555.666666666664,0.03625832108584767,0.9398,0,0.0
18223.166666666668,0.5400246924870822,0.17234,0,0.0


In [0]:
df.createOrReplaceTempView("credit_features")
display(spark.sql("""
  SELECT late_count, COUNT(*) AS customers, ROUND(AVG(label)*100,1) AS default_rate_pct
  FROM credit_features GROUP BY late_count ORDER BY late_count
"""))

late_count,customers,default_rate_pct
0,19931,11.7
1,4426,29.8
2,1899,38.8
3,1154,50.9
4,951,57.3
5,298,57.4
6,1341,70.3


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import mlflow

feature_cols = [c for c in df.columns if c not in ("ID", "label")]
train, test = df.randomSplit([0.8, 0.2], seed=42)

# Class weights to handle imbalance (about 78% non-default)
pos = train.filter("label = 1").count()
neg = train.filter("label = 0").count()
train = train.withColumn("w", F.when(F.col("label")==1, neg/pos).otherwise(1.0))

print(len(feature_cols), "features;", train.count(), "train rows;", test.count(), "test rows")

28 features; 24087 train rows; 5913 test rows


In [0]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

models = {
  "logistic_regression": LogisticRegression(featuresCol="features", labelCol="label", weightCol="w"),
  "gradient_boosted_trees": GBTClassifier(featuresCol="features", labelCol="label", weightCol="w", maxIter=50),
}

fitted = {}
for name, clf in models.items():
    with mlflow.start_run(run_name=name):
        model = Pipeline(stages=[assembler, clf]).fit(train)
        auc = evaluator.evaluate(model.transform(test))
        mlflow.log_param("model_type", name)
        mlflow.log_param("n_features", len(feature_cols))
        mlflow.log_metric("auc", auc)
        fitted[name] = model
        print(name, "AUC =", round(auc, 4))

logistic_regression AUC = 0.7526
gradient_boosted_trees AUC = 0.7821


In [0]:
best = fitted["gradient_boosted_trees"]
pred = best.transform(test)

cm = pred.groupBy("label", "prediction").count().orderBy("label", "prediction")
display(cm)

tp = pred.filter("label=1 AND prediction=1").count()
fp = pred.filter("label=0 AND prediction=1").count()
fn = pred.filter("label=1 AND prediction=0").count()
tn = pred.filter("label=0 AND prediction=0").count()

precision = tp/(tp+fp)
recall    = tp/(tp+fn)
accuracy  = (tp+tn)/(tp+tn+fp+fn)
print(f"Precision={precision:.3f}  Recall={recall:.3f}  Accuracy={accuracy:.3f}")

label,prediction,count
0.0,0.0,3720
0.0,1.0,930
1.0,0.0,484
1.0,1.0,779


Precision=0.456  Recall=0.617  Accuracy=0.761


In [0]:
imp = best.stages[-1].featureImportances
top = sorted(zip(feature_cols, imp.toArray()), key=lambda x: -x[1])[:10]
for f, v in top:
    print(f"{f:20s} {v:.3f}")

late_count           0.132
PAY_1                0.085
LIMIT_BAL            0.068
AGE                  0.062
pay_to_bill_ratio    0.060
utilisation          0.049
avg_payment          0.044
PAY_AMT3             0.041
PAY_AMT2             0.037
PAY_AMT1             0.037


In [0]:
with mlflow.start_run(run_name="gbt_final_eval"):
    mlflow.log_metric("auc", 0.7821)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("accuracy", accuracy)